# Lakehouse tables: Iceberg, Delta, Hudi

Each demo is one Kyuubi batch with its connector jars added (`--with iceberg`, `--with delta`,
`--with hudi`). Afterwards we query Iceberg and Delta interactively: `lab.sql(..., lake=True)`
opens a Kyuubi session as user `lake` whose engine starts with both connectors on.
(Hudi's Spark extension doesn't mix with the other two in one session, so Hudi is checked
from its job log.)

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

## Iceberg (catalog `ice`, stored in the Hive Metastore)

In [ ]:
r = lab.run('iceberg', run_id=RID)
lab.log_tail(r, '[')

In [ ]:
lab.sql("SELECT * FROM ice.lab.orders LIMIT 5", lake=True)

In [ ]:
snaps = lab.sql("SELECT committed_at, snapshot_id, operation FROM ice.lab.orders.snapshots ORDER BY committed_at", lake=True)
snaps

Time travel: the table as of its first snapshot.

In [ ]:
first = int(snaps["snapshot_id"].iloc[0])
lab.sql(f"""SELECT 'first snapshot' AS version, count(*) AS n FROM ice.lab.orders VERSION AS OF {first}
           UNION ALL SELECT 'now', count(*) FROM ice.lab.orders""", lake=True)

In [ ]:
lab.sql("SELECT * FROM ice.lab.orders.partitions", lake=True)

## Delta (path `s3a://warehouse/lake/delta/orders`, table `default.delta_orders`)

In [ ]:
r = lab.run('delta', run_id=RID)
lab.log_tail(r, '[')

In [ ]:
lab.sql("DESCRIBE HISTORY default.delta_orders", lake=True)[["version", "timestamp", "operation"]]

In [ ]:
lab.sql("""SELECT 'v0' AS version, count(*) n FROM default.delta_orders VERSION AS OF 0
           UNION ALL SELECT 'latest', count(*) FROM default.delta_orders""", lake=True)

Change data feed: what each version changed.

In [ ]:
h = lab.sql("DESCRIBE HISTORY default.delta_orders", lake=True)
# the feed exists from the commit after it was switched on
start = int(h[h["operation"] == "SET TBLPROPERTIES"]["version"].min()) + 1 if (h["operation"] == "SET TBLPROPERTIES").any() else 0
lab.sql(f"""SELECT _commit_version, _change_type, count(*) AS rows
           FROM table_changes('default.delta_orders', {start})
           GROUP BY 1, 2 ORDER BY 1, 2""", lake=True)

## Hudi (copy-on-write table at `s3a://warehouse/lake/hudi/orders_cow`)

In [ ]:
r = lab.run('hudi', run_id=RID)
lab.log_tail(r, '[')

In [ ]:
lab.s3_ls("warehouse", "lake/hudi/orders_cow/.hoodie/").head(15)